# Prerequisites

## Transaction 01 - Create Delta Table

In [0]:
%sql
USE CATALOG wns24082026;
DROP TABLE IF EXISTS quickstart_schema.users;
CREATE TABLE IF NOT EXISTS quickstart_schema.users(
    id INT, 
    name STRING, 
    dob DATE, 
    email STRING, 
    gender STRING, 
    country STRING, 
    region STRING, 
    city STRING, 
    asset INT, 
    marital_status STRING
);

DESCRIBE EXTENDED quickstart_schema.users

# Transaction 02 - Load 'users_001.csv' into delta table

In [0]:
df = spark.read.csv(
    path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

# Transaction 03 - Filter Country = 'India'

In [0]:
from pyspark.sql.functions import col
df.filter(col("country")=="India").write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

# Transaction 04 - Filter Country = 'United States'

In [0]:
df.filter(col("country")=="United States").write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

# List Transactions 

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "quickstart_schema.users")
delta_table.history().display()

In [0]:
delta_table.toDF().show()
display(delta_table.toDF())
spark.table("quickstart_schema.users").show()


# Read [Versioning]

## Pyspark

In [0]:
spark.read.option("versionAsOf",2).table("quickstart_schema.users").display()


## Sql

In [0]:
%sql
select * from quickstart_schema.users version as of 2

# Read [Timstamp]

In [0]:
spark.read.option("timestampAsOf","2026-08-28T06:10:33").table("quickstart_schema.users").display()

In [0]:
%sql
select * from quickstart_schema.users timestamp as of '2026-08-28T06:10:33'

## [Restore] Advantage

## Delta Table API

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark,"quickstart_schema.users")
delta_table.restoreToVersion(2)

In [0]:
spark.read.table("quickstart_schema.users").display()

## SQL

In [0]:
%sql
RESTORE TABLE quickstart_schema.users TO VERSION AS OF 1

In [0]:
%sql
select * from quickstart_schema.users